In [19]:
import hydra
import numpy as np
import pandas as pd
from pathlib import Path
import anndata as ad
from pathlib import Path
from omegaconf import DictConfig

# Path to the pkl file with new embeddings
PKL_PATH = Path("/home/matthew-mella/valinor/foundation-models-perturbation/molecule_embeddings/tahoe_PCA_emb.pkl")
# PKL_PATH = Path("/home/matthew-mella/valinor/foundation-models-perturbation/molecule_embeddings/tahoe_sci_op3_updated.pkl")



def get_valid_cids():
    """Load pkl and return set of valid pubchem_cids (compounds with LPM_emb)."""
    pkl_df = pd.read_pickle(PKL_PATH)
    tahoe_pkl = pkl_df[pkl_df["dataset"] == "tahoe"]
    tahoe_pkl_lpm = tahoe_pkl[tahoe_pkl["LPM_emb"].notna()].drop_duplicates(
        subset="pubchem_cid", keep="first"
    )
    return set(tahoe_pkl_lpm["pubchem_cid"].astype(int).values)


# def load_pkl_embeddings(emb_name):
#     """Load embedding lookup from pkl. Returns dict: pubchem_cid -> np.array."""
#     pkl_df = pd.read_pickle(PKL_PATH)

pkl_df = pd.read_pickle(PKL_PATH)
pkl_df

,cell_type,perturbagen,pert_type,is_control,pert_dose_uM,pert_time_h,pubchem_cid,perturbation_label,matched_l1000_idx,l1000_pert_dose_uM,l1000_pert_time_h,PCA.logFC,PCA.t,version,dim,l1000_phase,original_pert_name
0,CVCL_0320,alpelisib,compound,FALSE,5.0,24.0,56649450,alpelisib_5uM_24h,56649450_3_33uM_24h - 679_24h,3.33000,24.0,"[-6.29152089212975, -2.4460581799616086, -1.50...","[-25.571593719917146, 10.967763797756612, 5.94...",v1,64,2,Alpelisib
1,CVCL_0320,palbociclib,compound,FALSE,5.0,24.0,5330286,palbociclib_5uM_24h,5330286_3_3333uM_24h - 679_24h,3.33333,24.0,"[-13.19793165437994, -0.08249463076798, 0.5354...","[-53.2878789000218, -3.669506747244565, 1.5673...",v1,64,2,palbociclib
2,CVCL_0320,Infigratinib,compound,FALSE,5.0,24.0,53235510,Infigratinib_5uM_24h,53235510_3_33uM_24h - 679_24h,3.33000,24.0,"[4.480231322135845, -1.1060817904088174, -3.19...","[14.759168445466964, 3.1262594216205586, 13.08...",v1,64,2,Infigratinib
3,CVCL_0320,capivasertib,compound,FALSE,5.0,24.0,25227436,capivasertib_5uM_24h,25227436_3_3333uM_24h - 679_24h,3.33333,24.0,"[1.9403453409102045, -2.5021069117108006, 0.55...","[8.874716345253495, 8.004876061976683, -1.1839...",v1,64,2,Capivasertib
4,CVCL_0320,irinotecan,compound,FALSE,5.0,24.0,60838,irinotecan_5uM_24h,60838_3_3333uM_24h - 679_24h,3.33333,24.0,"[1.5883934302821328, -1.0127581498046516, 2.91...","[7.131042477606337, 3.923362131855394, -7.7857...",v1,64,2,Irinotecan
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1297,CVCL_0023,pimozide,compound,FALSE,5.0,24.0,16362,pimozide_5uM_24h,16362_10uM_24h - 679_24h,10.00000,24.0,"[-0.5108333021362381, -1.6709320754185015, -2....","[-2.2666823223152748, -5.863259267766386, 7.94...",v4,128,1,Pimozide
1298,CVCL_0023,gefitinib,compound,FALSE,5.0,24.0,123631,gefitinib_5uM_24h,123631_10uM_24h - 679_24h,10.00000,24.0,"[-4.435991623329821, -2.654278702691419, -1.40...","[-13.764374308952382, -11.255879690758508, 3.0...",v4,128,1,Gefitinib
1299,CVCL_0023,nimesulide,compound,FALSE,5.0,24.0,4495,nimesulide_5uM_24h,4495_10uM_24h - 679_24h,10.00000,24.0,"[0.7932697947000465, -0.9187910067682297, 0.32...","[4.343751859178172, -2.399327009425929, -2.933...",v4,128,1,Nimesulide
1300,CVCL_0023,Carbidopa (monohydrate),compound,FALSE,5.0,24.0,2563,Carbidopa (monohydrate)_5uM_24h,2563_10uM_24h - 679_24h,10.00000,24.0,"[-6.858054314866675, -2.390621724536526, 0.307...","[-10.140098574074862, -6.425200375933785, -10....",v4,128,1,Carbidopa (monohydrate)


In [26]:
v1 = pkl_df[pkl_df.version=='v1']
v1.reset_index(drop=True, inplace=True)
v1.l1000_pert_dose_uM.value_counts()
v1_single_dose = v1[v1.l1000_pert_dose_uM==3.33333]
print(v1_single_dose.pubchem_cid.nunique())
# Print number of unique cell lines
print(v1_single_dose.cell_type.value_counts())
# Print numer of unique cell lines for dosage '10'
v1_dose_10 = v1[v1.l1000_pert_dose_uM==10]
print(v1_dose_10.cell_type.value_counts())

101
cell_type
CVCL_0320    101
Name: count, dtype: int64
cell_type
CVCL_0023    75
Name: count, dtype: int64


In [11]:
pkl_df[pkl_df.version=='v1']['pubchem_cid'].nunique()

134